In [43]:
# Cell 1: Imports and Configurations

import os
import io
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr

# Display plots directly in the notebook
%matplotlib inline

# --- Directory Setup ---
# Set PROJECT_DIR to your base folder. Assuming notebook is in the project root.
PROJECT_DIR = os.getcwd() 
BUFFER_DIR = os.path.join(PROJECT_DIR, 'Buffer', 'temporal')
EVAL_DIR = os.path.join(PROJECT_DIR, 'Evaluation', 'temporal')
PREPROC_DIR = os.path.join(PROJECT_DIR, 'Preprocessing')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'timeseries_figures')

print(f"Project Directory: {PROJECT_DIR}")
print(f"Preprocessing Directory: {PREPROC_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Model Metadata ---
# Maps short codes to display names and plot colors
MODEL_MAP = {
    'rf': {'name': 'Random Forest', 'color': '#2ca02c'},
    'xgbr': {'name': 'XGBoost', 'color': '#9467bd'},
    'gbr': {'name': 'Gradient Boosting', 'color': '#d62728'},
    'lr': {'name': 'Linear Regression', 'color': '#1f77b4'},
    'm5p': {'name': 'M5P Model Tree', 'color': '#ff7f0e'}
}

Project Directory: c:\Users\mdabu\OneDrive\Desktop\practice\ML Lab\Final\ML Lab\Final Project (MOA)\Final MOA
Preprocessing Directory: c:\Users\mdabu\OneDrive\Desktop\practice\ML Lab\Final\ML Lab\Final Project (MOA)\Final MOA\Preprocessing
Output Directory: c:\Users\mdabu\OneDrive\Desktop\practice\ML Lab\Final\ML Lab\Final Project (MOA)\Final MOA\timeseries_figures


In [44]:
# Cell 2: Data Loading and Preprocessing

def parse_buffer_file(path):
    """Extracts the predictions table from raw Weka text logs."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
        
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        lines = f.readlines()
        
    # Find where the actual CSV data starts
    start_idx = next((i for i, line in enumerate(lines) if line.strip().startswith('inst#,actual')), None)
    if start_idx is None:
        raise ValueError("Header not found in Weka log.")
        
    # Read until the end of the data table (marked by '===')
    data_lines = [lines[start_idx]]
    for line in lines[start_idx + 1:]:
        if not line.strip() or line.strip().startswith('==='):
            break
        data_lines.append(line)
        
    # Convert text to DataFrame
    df = pd.read_csv(io.StringIO(''.join(data_lines)))
    df.columns = df.columns.str.strip()
    df['commodity_name'] = df['commodity_name'].astype(str).str.strip(" '")
    return df

# Load train/test references and convert date columns to datetime objects
train_ref = pd.read_csv(os.path.join(PREPROC_DIR, 'moa_train_80_lag.csv'))
test_ref = pd.read_csv(os.path.join(PREPROC_DIR, 'moa_test_20_lag.csv'))
train_ref['date'] = pd.to_datetime(train_ref[['year', 'month', 'day']])
test_ref['date'] = pd.to_datetime(test_ref[['year', 'month', 'day']])

# Identify where the test set begins chronologically
SPLIT_DATE = test_ref['date'].min()

train_path = os.path.join(EVAL_DIR, 'train', f"target_{7}", f"{"rf"}_buffer_train_{7}.csv")
test_path = os.path.join(EVAL_DIR, 'test', f"target_{7}", f"{"rf"}_buffer_{7}.csv")

print(f"{train_path=}")

if os.path.exists(train_path):
    print("The path exists!")
else:
    print("The path does not exist.")

print(f"{test_path=}")

if os.path.exists(test_path):
    print("The path exists!")
else:
    print("The path does not exist.")

def get_model_timeseries_df(model_key, horizon=7):
    """Merges actual and predicted data for a specific model and horizon."""
    target_col = f"target_{horizon}d"
    train_folder = f"target_{horizon}" if horizon != 30 else "target-30"
    
    # Define file paths
    train_path = os.path.join(EVAL_DIR, 'train', f"target_{horizon}", f"{model_key}_buffer_train_{horizon}.csv")
    test_path = os.path.join(EVAL_DIR, 'test', f"target_{horizon}", f"{model_key}_buffer_{horizon}.csv")

    print(f"{train_path=}")
    print(f"{test_path=}")

    # Parse data logs
    train_buf = parse_buffer_file(train_path)
    test_buf = parse_buffer_file(test_path)
    
    # Format Train Data
    df_train = train_ref.copy()
    df_train['predicted'] = train_buf['predicted']
    df_train['actual_target'] = df_train[target_col]
    df_train['split'] = 'Train'
    
    # Format Test Data
    df_test = test_ref.copy()
    df_test['predicted'] = test_buf['predicted']
    df_test['actual_target'] = df_test[target_col]
    df_test['split'] = 'Test'
    
    return pd.concat([df_train, df_test], ignore_index=True)

train_path='c:\\Users\\mdabu\\OneDrive\\Desktop\\practice\\ML Lab\\Final\\ML Lab\\Final Project (MOA)\\Final MOA\\Evaluation\\temporal\\train\\target_7\\rf_buffer_train_7.csv'
The path exists!
test_path='c:\\Users\\mdabu\\OneDrive\\Desktop\\practice\\ML Lab\\Final\\ML Lab\\Final Project (MOA)\\Final MOA\\Evaluation\\temporal\\test\\target_7\\rf_buffer_7.csv'
The path exists!


In [45]:
# Cell 3: Main Plotting Function for Single Commodities

def plot_commodity_timeseries(model_key='rf', horizon=7, commodity='Rice - Medium', save=True):
    """Plots actual vs predicted prices over time for a single commodity."""
    model_name = MODEL_MAP[model_key]['name']
    model_color = MODEL_MAP[model_key]['color']
    
    # Get combined train/test dataframe
    df = get_model_timeseries_df(model_key, horizon)
    sub = df[df['commodity_name'] == commodity].copy()
    if sub.empty: return None
    
    unit = sub['retail_unit'].iloc[0]
    
    # Calculate daily averages
    daily = sub.groupby(['date', 'split'], as_index=False).agg(
        {'actual_target': 'mean', 'predicted': 'mean'}
    ).sort_values('date')
    
    # Calculate evaluation metrics on the Test set
    test_data = daily[daily['split'] == 'Test']
    if len(test_data) > 1:
        act, pred = test_data['actual_target'].values, test_data['predicted'].values
        rmse = np.sqrt(np.mean((act - pred) ** 2))
        mape = np.mean(np.abs((act - pred) / act)) * 100
        r_val, _ = pearsonr(act, pred) if np.std(act) > 0 and np.std(pred) > 0 else (np.nan, 0)
        metrics = f"Test Metrics:\nPearson r: {r_val:.4f}\nRMSE: {rmse:.2f}\nMAPE: {mape:.2f}%"
    else:
        metrics = "Test Metrics: N/A"
        
    # --- Plotting Setup ---
    fig, ax = plt.subplots(figsize=(13, 5.5), dpi=150)
    
    # Draw Lines
    ax.plot(daily['date'], daily['actual_target'], label='Actual Price', color='#1f77b4', lw=2)
    ax.plot(daily['date'], daily['predicted'], label=f'Predicted ({model_name})', color=model_color, lw=1.8, ls='--')
    
    # Draw Background Regions (Train vs Test)
    ax.axvspan(daily['date'].min(), SPLIT_DATE, color='#2ca02c', alpha=0.1, label='Train Period')
    ax.axvspan(SPLIT_DATE, daily['date'].max(), color='#ff7f0e', alpha=0.12, label='Test Period')
    ax.axvline(SPLIT_DATE, color='k', ls=':', lw=1.6, label=f'Split Date ({SPLIT_DATE.date()})')
    
    # Add Metrics Text Box
    props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='#bbb')
    ax.text(0.985, 0.05, metrics, transform=ax.transAxes, va='bottom', ha='right', bbox=props, fontsize=9.5)
    
    # Formatting
    ax.set_title(f'{commodity} - {model_name} ({horizon}-Day Forecast)', fontsize=12, fontweight='bold', pad=12)
    ax.set_ylabel(f'Price (BDT / {unit})')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    fig.autofmt_xdate()
    ax.grid(True, ls='--', alpha=0.5)
    ax.legend(loc='upper left', fontsize=9)
    fig.tight_layout()
    
    # Save Output
    if save:
        clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', commodity)
        save_dir = os.path.join(OUTPUT_DIR, "Per Commodity Plot", model_key, f"target_{horizon}")
        os.makedirs(save_dir, exist_ok=True)
        out_path = os.path.join(save_dir, f"{model_key}_{clean_name}_{horizon}d.png")
        fig.savefig(out_path, bbox_inches='tight')
        print(f"Saved: {out_path}")
        
    plt.show() # Display inline

In [46]:
# Cell 4: Generating Cross-Model Comparisons (FIXED)

def generate_cross_model_comparison(commodity='Rice - Medium', horizon=7):
    """Generates a stacked subplot comparing all models for a specific commodity."""
    
    # 1. First check if data actually exists for this commodity across models
    has_data = False
    for m_key in MODEL_MAP.keys():
        df = get_model_timeseries_df(m_key, horizon)
        if not df[df['commodity_name'] == commodity].empty:
            has_data = True
            break
            
    if not has_data:
        print(f"Skipping {commodity}: No data found for horizon {horizon}.")
        return

    print(f"Generating Cross-Model Comparison for {commodity}...")
    fig, axes = plt.subplots(len(MODEL_MAP), 1, figsize=(14, 18), sharex=True, dpi=140)
    
    # Ensure axes is iterable even if len(MODEL_MAP) == 1
    if len(MODEL_MAP) == 1:
        axes = [axes]
        
    last_valid_daily = None

    for idx, (m_key, m_info) in enumerate(MODEL_MAP.items()):
        ax = axes[idx]
        df = get_model_timeseries_df(m_key, horizon)
        sub = df[df['commodity_name'] == commodity]
        
        if sub.empty: 
            ax.text(0.5, 0.5, "No Data Available", ha='center', va='center')
            continue
            
        unit = sub['retail_unit'].iloc[0]
        daily = sub.groupby(['date', 'split'], as_index=False).agg(
            {'actual_target': 'mean', 'predicted': 'mean'}
        ).sort_values('date')
        last_valid_daily = daily
        
        # Calculate brief metrics
        test_data = daily[daily['split'] == 'Test']
        if not test_data.empty and len(test_data) > 1:
            r_val, _ = pearsonr(test_data['actual_target'], test_data['predicted'])
            rmse = np.sqrt(np.mean((test_data['actual_target'] - test_data['predicted']) ** 2))
            title_suffix = f'(Test r = {r_val:.3f}, RMSE = {rmse:.2f})'
        else:
            title_suffix = '(Insufficient Test Data)'
        
        # Plot lines and backgrounds
        ax.plot(daily['date'], daily['actual_target'], label='Actual', color='#1f77b4', lw=1.8)
        ax.plot(daily['date'], daily['predicted'], label=m_info['name'], color=m_info['color'], lw=1.6, ls='--')
        ax.axvspan(daily['date'].min(), SPLIT_DATE, color='#2ca02c', alpha=0.08)
        ax.axvspan(SPLIT_DATE, daily['date'].max(), color='#ff7f0e', alpha=0.10)
        ax.axvline(SPLIT_DATE, color='k', ls=':', lw=1.2)
        
        ax.set_ylabel(f'BDT / {unit}')
        ax.set_title(f'{m_info["name"]} {title_suffix}', fontweight='bold', loc='left')
        ax.legend(loc='upper left', fontsize=8.5)
        ax.grid(True, ls='--', alpha=0.5)

    # Format X axis using the last successfully processed dataset
    if last_valid_daily is not None:
        axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        fig.autofmt_xdate()
        
    fig.suptitle(f'Cross-Model Comparison: {commodity} ({horizon}-Day Forecast)', fontsize=14, fontweight='bold', y=0.995)
    fig.tight_layout()
    
    # Save Output with dynamic directory creation
    clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', commodity)
    out_dir = os.path.join(OUTPUT_DIR, "Cross-Model", f"target_{horizon}")
    os.makedirs(out_dir, exist_ok=True)  # <-- Fixed directory bug
    
    out_path = os.path.join(out_dir, f"cross_model_{clean_name}_{horizon}d.png")
    fig.savefig(out_path, bbox_inches='tight')
    plt.show()
    plt.close(fig) # Free up memory

In [47]:
# Cell 5: Generating Multi-Horizon Comparisons

def generate_multi_horizon_comparison(model_key='rf', commodity='Soybean Oil(loose)'):
    """Compares 7-day, 14-day, and 30-day forecast horizons for a single model."""
    print(f"Generating Multi-Horizon Comparison for {commodity}...")
    horizons = [7, 14, 30]
    fig, axes = plt.subplots(len(horizons), 1, figsize=(14, 11), sharex=True, dpi=140)
    
    for idx, h in enumerate(horizons):
        ax = axes[idx]
        df = get_model_timeseries_df(model_key, h)
        sub = df[df['commodity_name'] == commodity]
        if sub.empty: continue
            
        unit = sub['retail_unit'].iloc[0]
        daily = sub.groupby(['date', 'split'], as_index=False).agg(
            {'actual_target': 'mean', 'predicted': 'mean'}
        ).sort_values('date')
        
        # Calculate brief metrics
        test_data = daily[daily['split'] == 'Test']
        r_val, _ = pearsonr(test_data['actual_target'], test_data['predicted'])
        rmse = np.sqrt(np.mean((test_data['actual_target'] - test_data['predicted']) ** 2))
        
        # Plot lines and backgrounds
        ax.plot(daily['date'], daily['actual_target'], label='Actual', color='#1f77b4', lw=1.8)
        ax.plot(daily['date'], daily['predicted'], label=f'{h}-Day Forecast', color='#2ca02c', lw=1.6, ls='--')
        ax.axvspan(daily['date'].min(), SPLIT_DATE, color='#2ca02c', alpha=0.08)
        ax.axvspan(SPLIT_DATE, daily['date'].max(), color='#ff7f0e', alpha=0.10)
        ax.axvline(SPLIT_DATE, color='k', ls=':', lw=1.2)
        
        ax.set_ylabel(f'BDT / {unit}')
        ax.set_title(f'{h}-Day Horizon (Test r = {r_val:.3f}, RMSE = {rmse:.2f})', fontweight='bold', loc='left')
        ax.legend(loc='upper left', fontsize=8.5)
        ax.grid(True, ls='--', alpha=0.5)

    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    fig.autofmt_xdate()
    
    model_name = MODEL_MAP[model_key]['name']
    fig.suptitle(f'{model_name}: Multi-Horizon Comparison - {commodity}', fontsize=14, fontweight='bold', y=0.995)
    fig.tight_layout()
    
    # Save Output

    clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', commodity)
    out_dir = os.path.join(OUTPUT_DIR, f"Multi Horizon", f"{model_key}")
    os.makedirs(out_dir, exist_ok=True)
    
    out_path = os.path.join(out_dir, f"{model_key}_multi_horizon_{clean_name}.png")
    fig.savefig(out_path, bbox_inches='tight')
    plt.show()

# Consolidated Report-Ready Visualizations\n
\n
The sections below solve the **414-image explosion problem** by combining all 18 commodities, 5 models, and 3 horizons into a compact, publication-ready structure:\n
- **Cell 6:** Macro Grid (All 18 Commodities on 1 Page)\n
- **Cell 7:** Thematic Category Grids (5 Figures covering all 18 commodities)\n
- **Cell 8:** Single-Axis Cross-Model Overlay (All 5 models on 1 plot)\n
- **Cell 9:** Optimized Report-Ready Execution Runner\n

In [48]:
# Cell 6: Consolidated Multi-Commodity Macro Grid (All 18 Commodities on 1 Page)

def generate_all_commodities_macro_grid(model_key='rf', horizon=7, save=True, show=True):
    """
    Consolidates ALL 18 commodities into a single 6x3 small-multiples grid figure.
    Perfect for a full-page appendix figure in your academic report without figure clutter.
    """
    model_name = MODEL_MAP[model_key]['name']
    model_color = MODEL_MAP[model_key]['color']
    full_df = get_model_timeseries_df(model_key, horizon)
    products = sorted(full_df['commodity_name'].unique().tolist())
    
    fig, axes = plt.subplots(6, 3, figsize=(18, 22), dpi=140, sharex=True)
    axes = axes.flatten()
    
    for idx, prod in enumerate(products):
        ax = axes[idx]
        sub = full_df[full_df['commodity_name'] == prod]
        unit = sub['retail_unit'].iloc[0]
        daily = sub.groupby(['date', 'split'], as_index=False).agg({
            'actual_target': 'mean',
            'predicted': 'mean'
        }).sort_values('date')
        
        test_data = daily[daily['split'] == 'Test']
        if len(test_data) > 1:
            act, pred = test_data['actual_target'].values, test_data['predicted'].values
            r_val, _ = pearsonr(act, pred) if np.std(act) > 0 and np.std(pred) > 0 else (np.nan, 0)
            rmse = np.sqrt(np.mean((act - pred) ** 2))
            metric_tag = f"Test r={r_val:.2f}, RMSE={rmse:.1f}"
        else:
            metric_tag = ""
            
        ax.plot(daily['date'], daily['actual_target'], color='#1f77b4', lw=1.3, label='Actual' if idx == 0 else "")
        ax.plot(daily['date'], daily['predicted'], color=model_color, lw=1.2, ls='--', label=f'Pred ({model_name})' if idx == 0 else "")
        ax.axvspan(daily['date'].min(), SPLIT_DATE, color='#2ca02c', alpha=0.08, label='Train (80%)' if idx == 0 else "")
        ax.axvspan(SPLIT_DATE, daily['date'].max(), color='#ff7f0e', alpha=0.10, label='Test (20%)' if idx == 0 else "")
        ax.axvline(SPLIT_DATE, color='#444444', ls=':', lw=1.0)
        
        ax.set_title(f"{prod} ({unit}) | {metric_tag}", fontsize=9.5, fontweight='bold', pad=4)
        ax.grid(True, ls='--', alpha=0.4)
        ax.tick_params(axis='both', which='major', labelsize=8)
        
    for ax in axes[-3:]:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
        
    fig.autofmt_xdate()
    fig.suptitle(f"BD-FPP Macro Overview: All 18 Commodities -- {model_name} ({horizon}-Day Target Horizon)", 
                 fontsize=15, fontweight='bold', y=0.995)
    
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.985), ncol=4, fontsize=10, frameon=True)
    fig.tight_layout(rect=[0, 0, 1, 0.975])
    
    if save:
        report_dir = os.path.join(OUTPUT_DIR, "Report_Ready_Figures")
        os.makedirs(report_dir, exist_ok=True)
        out_path = os.path.join(report_dir, f"macro_grid_all_18_{model_key}_{horizon}d.png")
        fig.savefig(out_path, bbox_inches='tight')
        print(f"Saved Consolidated Macro Grid: {out_path}")
        
    if show:
        plt.show()
    else:
        plt.close(fig)

In [49]:
# Cell 7: Thematic Category Grids (5 Groups Covering All 18 Commodities)

CATEGORIES = {
    "Grains_and_Flour": {
        "title": "Grains & Flour Staples (Cereals)",
        "items": ["Rice - Fine", "Rice - Medium", "Ata (Packet)", "Ata (loose) - White"],
        "layout": (2, 2)
    },
    "Meat_Fish_and_Poultry": {
        "title": "Protein: Meat, Poultry & Fish",
        "items": ["Beef", "Broiler chicken", "Egg Farm-Red", "Pangash (big)"],
        "layout": (2, 2)
    },
    "Edible_Oils_and_Dairy": {
        "title": "Cooking Oils & Dairy Essentials",
        "items": ["Soybean Oil(loose)", "Palm Oil", "Milk"],
        "layout": (3, 1)
    },
    "Produce_and_Pulses": {
        "title": "Fresh Vegetables, Tubers & Lentils",
        "items": ["Potato (Holland) - Red", "Green Chili (Local)", "Lentils - Desi-Whole"],
        "layout": (3, 1)
    },
    "Spices_and_Aromatics": {
        "title": "Spices & Aromatics (High Volatility)",
        "items": ["Garlic (Imported)", "Garlic (local) - Big Size", "Ginger (Imported)", "Onion (local)"],
        "layout": (2, 2)
    }
}

def generate_category_grids(model_key='rf', horizon=7, save=True, show=True):
    """
    Generates 5 category figures covering all 18 commodities grouped by food sector.
    Reduces 18 individual figures down to 5 publication-ready multi-panel figures.
    """
    model_name = MODEL_MAP[model_key]['name']
    model_color = MODEL_MAP[model_key]['color']
    full_df = get_model_timeseries_df(model_key, horizon)
    
    report_dir = os.path.join(OUTPUT_DIR, "Report_Ready_Figures")
    os.makedirs(report_dir, exist_ok=True)
    
    for cat_key, cat in CATEGORIES.items():
        rows, cols = cat['layout']
        items = cat['items']
        
        fig, axes = plt.subplots(rows, cols, figsize=(15 if cols == 2 else 14, 8 if rows == 2 else 10), dpi=140, sharex=True)
        axes = np.array(axes).flatten()
        
        for idx, prod in enumerate(items):
            ax = axes[idx]
            sub = full_df[full_df['commodity_name'] == prod]
            unit = sub['retail_unit'].iloc[0]
            daily = sub.groupby(['date', 'split'], as_index=False).agg({
                'actual_target': 'mean',
                'predicted': 'mean'
            }).sort_values('date')
            
            test_data = daily[daily['split'] == 'Test']
            r_val, _ = pearsonr(test_data['actual_target'], test_data['predicted'])
            rmse = np.sqrt(np.mean((test_data['actual_target'] - test_data['predicted']) ** 2))
            mape = np.mean(np.abs((test_data['actual_target'] - test_data['predicted']) / test_data['actual_target'])) * 100
            
            ax.plot(daily['date'], daily['actual_target'], color='#1f77b4', lw=1.8, label='Actual Price')
            ax.plot(daily['date'], daily['predicted'], color=model_color, lw=1.6, ls='--', label=f'Pred ({model_name})')
            ax.axvspan(daily['date'].min(), SPLIT_DATE, color='#2ca02c', alpha=0.08, label='Train (80%)' if idx == 0 else "")
            ax.axvspan(SPLIT_DATE, daily['date'].max(), color='#ff7f0e', alpha=0.10, label='Test (20%)' if idx == 0 else "")
            ax.axvline(SPLIT_DATE, color='#444444', ls=':', lw=1.2)
            
            ax.set_title(f"{prod} ({unit}) | Test r={r_val:.3f}, RMSE={rmse:.2f}, MAPE={mape:.1f}%", 
                         fontsize=10.5, fontweight='bold', pad=6)
            ax.set_ylabel(f"Price (BDT / {unit})", fontsize=9.5)
            ax.grid(True, ls='--', alpha=0.5)
            ax.legend(loc='upper left', fontsize=8.5)
            
        for ax in axes[-cols:]:
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        fig.autofmt_xdate()
        
        fig.suptitle(f"{cat['title']} -- {model_name} ({horizon}-Day Forecast Horizon)", 
                     fontsize=13, fontweight='bold', y=0.995)
        fig.tight_layout()
        
        if save:
            out_path = os.path.join(report_dir, f"category_{cat_key}_{model_key}_{horizon}d.png")
            fig.savefig(out_path, bbox_inches='tight')
            print(f"Saved Category Grid: {out_path}")
            
        if show:
            plt.show()
        else:
            plt.close(fig)

In [50]:
# Cell 8: Compact Single-Axis Cross-Model Overlay (All 5 Models on 1 Plot)

def generate_cross_model_overlay(commodity='Rice - Medium', horizon=7, save=True, show=True):
    """
    Plots the ground truth Actual price and overlays all 5 regression models on a SINGLE axis.
    Replaces tall stacked multi-panel figures with a compact, report-friendly 5-inch diagram.
    """
    fig, ax = plt.subplots(figsize=(13, 5.5), dpi=150)
    
    for idx, (m_key, m_info) in enumerate(MODEL_MAP.items()):
        full_df = get_model_timeseries_df(m_key, horizon)
        sub = full_df[full_df['commodity_name'] == commodity]
        if sub.empty: continue
        unit = sub['retail_unit'].iloc[0]
        daily = sub.groupby(['date', 'split'], as_index=False).agg({
            'actual_target': 'mean',
            'predicted': 'mean'
        }).sort_values('date')
        
        if idx == 0:
            ax.plot(daily['date'], daily['actual_target'], color='#1f77b4', lw=2.2, label='Actual Market Price (Ground Truth)', zorder=5)
            min_date = daily['date'].min()
            max_date = daily['date'].max()
            ax.axvspan(min_date, SPLIT_DATE, color='#2ca02c', alpha=0.08, label='Train Period (80%)')
            ax.axvspan(SPLIT_DATE, max_date, color='#ff7f0e', alpha=0.10, label='Test Period (20%)')
            ax.axvline(SPLIT_DATE, color='#333333', ls=':', lw=1.6, label=f'Split Date ({SPLIT_DATE.strftime("%Y-%m-%d")})')
            
        test_data = daily[daily['split'] == 'Test']
        r_val, _ = pearsonr(test_data['actual_target'], test_data['predicted'])
        rmse = np.sqrt(np.mean((test_data['actual_target'] - test_data['predicted']) ** 2))
        
        style = '--' if m_key in ['rf', 'xgbr'] else (':' if m_key == 'gbr' else '-.')
        lw = 1.8 if m_key in ['rf', 'xgbr'] else 1.3
        alpha = 0.95 if m_key in ['rf', 'xgbr'] else 0.75
        
        ax.plot(daily['date'], daily['predicted'], color=m_info['color'], lw=lw, ls=style, alpha=alpha,
                label=f"{m_info['name']} (r={r_val:.3f}, RMSE={rmse:.2f})")
                
    ax.set_title(f"Cross-Model Single-Axis Overlay: {commodity} ({unit}) -- {horizon}-Day Forecast Horizon",
                 fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel('Time (Calendar Date)', fontsize=10.5, labelpad=8)
    ax.set_ylabel(f'Price (BDT / {unit})', fontsize=10.5, labelpad=8)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    fig.autofmt_xdate()
    ax.grid(True, ls='--', alpha=0.5)
    ax.legend(loc='upper left', framealpha=0.92, fontsize=8.5)
    fig.tight_layout()
    
    if save:
        report_dir = os.path.join(OUTPUT_DIR, "Report_Ready_Figures")
        os.makedirs(report_dir, exist_ok=True)
        clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', commodity)
        out_path = os.path.join(report_dir, f"overlay_all_models_{clean_name}_{horizon}d.png")
        fig.savefig(out_path, bbox_inches='tight')
        print(f"Saved Overlay: {out_path}")
        
    if show:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
# Cell 9: Optimized Execution Runner (Compact, Report-Ready Figures)

print("==========================================================================")
print("1. GENERATING MACRO GRID (ALL 18 COMMODITIES ON 1 FULL-PAGE FIGURE)")
print("==========================================================================")
# Generates 1 single figure for Random Forest at 7-day horizon covering ALL 18 commodities
for model in MODEL_MAP.keys():
    for horizon in [7, 14, 30]:
        generate_all_commodities_macro_grid(model_key=model, horizon=horizon, save=True, show=True)

print("==========================================================================")
print("2. GENERATING 5 THEMATIC CATEGORY GRIDS (COVERING ALL 18 COMMODITIES)")
print("==========================================================================")
# Generates 5 clean grouped figures (Cereals, Protein, Oils, Vegetables, Spices)
for model in MODEL_MAP.keys():
    for horizon in [7, 14, 30]:
        generate_category_grids(model_key=model, horizon=horizon, save=True, show=True)

print("==========================================================================")
print("3. GENERATING CROSS-MODEL SINGLE-AXIS OVERLAYS (KEY COMMODITIES)")
print("==========================================================================")
# Direct model comparison on single compact plots for contrasting market commodities
for staple in ['Rice - Medium', 'Soybean Oil(loose)', 'Onion (local)']:
    for horizon in [7, 14, 30]:
        generate_cross_model_overlay(commodity=staple, horizon=horizon, save=True, show=True)

print("==========================================================================")
print("4. BATCH DISK EXPORT (OPTIONAL: SAVES TO DISK WITHOUT NOTEBOOK BLOAT)")
print("==========================================================================")
# To export all individual 270 plots to disk WITHOUT bloating your notebook memory:
def batch_export_individual_plots_to_disk(models=['rf', 'xgbr', 'gbr', 'lr', 'm5p'], horizons=[7, 14, 30]):
    all_products = sorted(train_ref['commodity_name'].unique().tolist())
    print(f"Exporting individual plots to disk for {len(all_products)} products...")
    for m in models:
        for h in horizons:
            for p in all_products:
                # Setting save=True and show=False keeps notebook clean (<100KB)
                plot_commodity_timeseries(model_key=m, horizon=h, commodity=p, save=True)
                plt.close('all') # Essential to free memory
    print("Individual plots exported cleanly to disk.")

# Uncomment the line below if you wish to run the full disk export:
# batch_export_individual_plots_to_disk(models=['rf', 'xgbr', 'gbr'], horizons=[7])
print("Report-ready figures saved to: timeseries_figures/Report_Ready_Figures/")

## (Optional / Legacy) Brute-Force Single-Plot Loops\n
*Note: Running all nested loops generates 414 individual plots which bloats the notebook to ~43MB. Use Cell 9 above for report figures or batch disk export.*

In [ ]:
# # Cell 6: Run Plot Generators

products = [
    "Ata (Packet)",
    "Ata (loose) - White",
    "Beef",
    "Broiler chicken",
    "Egg Farm-Red",
    "Garlic (Imported)",
    "Garlic (local) - Big Size",
    "Ginger (Imported)",
    "Green Chili (Local)",
    "Lentils - Desi-Whole",
    "Milk",
    "Onion (local)",
    "Palm Oil",
    "Pangash (big)",
    "Potato (Holland) - Red",
    "Rice - Fine",
    "Rice - Medium",
    "Soybean Oil(loose)"
]
# print(len(products))

print("--- 1. Generating Staple Diagrams for Random Forest (7-day) ---")
for product in products:
    for horizon in [7, 14, 30]:
        for model_key in MODEL_MAP.keys():
            plot_commodity_timeseries(model_key=model_key, horizon=horizon, commodity=product)

print("--- 2. Generating Cross-Model 5-Panel Comparison ---")
for product in products:
    for horizon in [7, 14, 30]:
        generate_cross_model_comparison(commodity=product, horizon=horizon)

print("--- 3. Generating Multi-Horizon Comparison ---")
for product in products:
    for model in MODEL_MAP.keys():
        generate_multi_horizon_comparison(model_key=model, commodity=product)